In [23]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [24]:
#!pip install groq
#!pip install autogen

In [25]:
import os
from dotenv import load_dotenv
import pprint
env_path = r'.env'
load_dotenv(env_path)

True

In [32]:
# Configure - note code updated here,
config_list = [{
    "model": "llama-3.1-8b-instant",
    "api_key": os.environ.get("GROQ_API_KEY"),
    "base_url": "https://api.groq.com/openai/v1"
}]


In [33]:
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent
from autogen import initiate_chats

agent = ConversableAgent(
    name="chatbot",
    llm_config={"config_list": config_list},
    human_input_mode="NEVER",
)

In [34]:
reply = agent.generate_reply(
    messages=[{"content": "Tell me a joke.", "role": "user"}]
)
# note code updated here,
print(reply)


[autogen.oai.client: 05-11 20:55:30] {746} WARNING - Model llama-3.1-8b-instant is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
A man walked into a library and asked the librarian, "Do you have any books on Pavlov's dogs and Schrödinger's cat?" The librarian replied, "It rings a bell, but I'm not sure if it's here or not."


In [35]:
# Define the RegistrationAgent for collecting user details
registration_agent = ConversableAgent(
    name="User_Registration_Agent",
    system_message='''You are a friendly registration assistant.
    Your task is to collect the user's full name, company name, and the product they are interested in.
    Do not ask for additional details. Respond with 'END_SESSION' once all details are obtained.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [36]:
# Define the HelpDeskAgent for troubleshooting
help_desk_agent = AssistantAgent(
    name="Help_Desk_Agent",
    system_message='''You are a knowledgeable support assistant.
    Your role is to assist users in buying our product.
    Offer clear steps, share useful resources, and escalate if required.
    Respond with 'END_SESSION' once the issue is addressed.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [37]:
# Define the SalesAgent for product recommendations
sales_agent = AssistantAgent(
    name="Sales_Recommendation_Agent",
    system_message='''You are a sales assistant.
    Based on user information, suggest the most suitable product and provide additional details.
    If the user expresses interest, proceed with gathering purchase intent details.
    Respond with 'END_SESSION' when user decides to proceed or not.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [38]:
# Define the RelayAgent for communication handling
relay_agent = UserProxyAgent(
    name="Interaction_Relay_Agent",
    system_message='''You act as a bridge between users and specialized agents.
    You do not process requests yourself but ensure smooth communication.''',
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
)

In [40]:
conversation_flow = [
    {
        "sender": registration_agent,
        "recipient": relay_agent,
        "message": "Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Extract user details as JSON: {'full_name': '', 'company': '', 'product': ''}",
        },
        "max_turns": 2,
        "clear_history": True
    },
    {
        "sender": relay_agent,
        "recipient": sales_agent,
        "message": "Based on user details, suggest a suitable product.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize recommended products as JSON: {'recommended_product': '', 'justification': ''}",
        },
        "max_turns": 3,
        "clear_history": False
    },
    {
        "sender": relay_agent,
        "recipient": help_desk_agent,
        "message": "Based on sale recommendation, closen the conversation with formal sale pitch",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize the assistance provided as JSON: {'resolution_steps': []}",
        },
        "max_turns": 3,
        "clear_history": False
    }
]

# Execute the conversations
session_results = initiate_chats(conversation_flow)
pprint.pp(session_results)


********************************************************************************
Starting a new chat....

********************************************************************************
User_Registration_Agent (to Interaction_Relay_Agent):

Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.

--------------------------------------------------------------------------------
Interaction_Relay_Agent (to User_Registration_Agent):

Sridurga Bank Of America Office 365

--------------------------------------------------------------------------------
[autogen.oai.client: 05-11 21:00:47] {746} WARNING - Model llama-3.1-8b-instant is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
User_Registration_Agent (to Interaction_Relay_Agent):

Thank you for the information. However, I need a more detailed response for the full name. C